In [1]:
import sys, os
os.chdir("/home/dsa/new_seg_final/eomt")

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image, ImageDraw
from torchvision import tv_tensors

print("Imports OK, device:", "cuda" if torch.cuda.is_available() else "cpu")

Imports OK, device: cuda


In [ ]:
# --- Config ---
CKPT_PATH = "runs/coronary_instance_eomt_small_512_dinov2/version_0/checkpoints/best.ckpt"
DATA_ROOT = Path("/home/dsa/new_seg_final/single_dataset")
IMG_DIR = DATA_ROOT / "test" / "images"
LABEL_DIR = DATA_ROOT / "test" / "labels"
NUM_SAMPLES = 50
IMG_SIZE = (512, 512)
NUM_CLASSES = 9
EVAL_TOP_K = 100
SCORE_THRESH = 0.3

CLASS_NAMES = {
    0: "lad", 1: "lm", 2: "lcx", 3: "lad_b", 4: "lcx_b",
    5: "inter", 6: "rca", 7: "pda", 8: "pborca",
}

CLASS_COLORS = [
    (1.0, 0.0, 0.0),     # lad - red
    (0.0, 1.0, 0.0),     # lm - green
    (0.0, 0.0, 1.0),     # lcx - blue
    (1.0, 1.0, 0.0),     # lad_b - yellow
    (1.0, 0.0, 1.0),     # lcx_b - magenta
    (0.0, 1.0, 1.0),     # inter - cyan
    (1.0, 0.5, 0.0),     # rca - orange
    (0.5, 0.0, 1.0),     # pda - purple
    (0.0, 0.5, 0.0),     # pborca - dark green
]
print("Config OK")

Config OK


In [4]:
# --- Load model ---
from training.mask_classification_instance import MaskClassificationInstance
from models.eomt import EoMT
from models.vit import ViT

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

encoder = ViT(
    backbone_name="vit_small_patch14_dinov2.lvd142m",
    img_size=IMG_SIZE,
    patch_size=14,
)

network = EoMT(
    encoder=encoder,
    num_classes=NUM_CLASSES,
    img_size=IMG_SIZE,
    num_q=100,
    num_blocks=3,
)

model = MaskClassificationInstance(
    network=network,
    img_size=IMG_SIZE,
    num_classes=NUM_CLASSES,
    attn_mask_annealing_enabled=True,
    attn_mask_annealing_start_steps=[2960, 7400, 11840],
    attn_mask_annealing_end_steps=[7400, 11840, 14800],
)

state_dict = ckpt["state_dict"]
state_dict = {k.replace("._orig_mod.", "."): v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)
model.eval()
model.cuda()
print("Model loaded!")

TypeError: EoMT.__init__() got an unexpected keyword argument 'img_size'

In [ ]:
# --- Helper functions ---

def load_gt(label_path, h, w):
    """Load GT masks and labels from YOLO-format label file."""
    masks, labels = [], []
    if not label_path.exists() or label_path.stat().st_size == 0:
        return masks, labels
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            class_id = int(parts[0])
            coords = list(map(float, parts[1:]))
            polygon = []
            for i in range(0, len(coords) - 1, 2):
                polygon.append((coords[i] * w, coords[i + 1] * h))
            if len(polygon) < 3:
                continue
            mask_img = Image.new("L", (w, h), 0)
            ImageDraw.Draw(mask_img).polygon(polygon, fill=1)
            mask = np.array(mask_img, dtype=bool)
            if not mask.any():
                continue
            masks.append(mask)
            labels.append(class_id)
    return masks, labels


def render_masks_on_image(img_np, masks, labels, alpha=0.5):
    """Overlay colored instance masks on image."""
    overlay = img_np.copy().astype(np.float32)
    for mask, label in zip(masks, labels):
        color = np.array(CLASS_COLORS[label]) * 255.0
        for c in range(3):
            overlay[:, :, c] = np.where(
                mask, overlay[:, :, c] * (1 - alpha) + color[c] * alpha, overlay[:, :, c]
            )
    return np.clip(overlay, 0, 255).astype(np.uint8)


def scale_img_size(size):
    factor = min(IMG_SIZE[0] / size[0], IMG_SIZE[1] / size[1])
    return [round(s * factor) for s in size]


@torch.no_grad()
def predict(img_tensor):
    """Run inference, return (masks, labels, scores)."""
    device = next(model.parameters()).device
    img = img_tensor.to(device)
    orig_h, orig_w = img.shape[-2], img.shape[-1]
    new_h, new_w = scale_img_size((orig_h, orig_w))

    pil_img = Image.fromarray(img.permute(1, 2, 0).cpu().numpy())
    pil_img = pil_img.resize((new_w, new_h), Image.BILINEAR)
    resized = torch.from_numpy(np.array(pil_img)).permute(2, 0, 1).to(device)

    pad_h = max(0, IMG_SIZE[0] - resized.shape[-2])
    pad_w = max(0, IMG_SIZE[1] - resized.shape[-1])
    padded = F.pad(resized, [0, pad_w, 0, pad_h])

    with torch.cuda.amp.autocast():
        mask_logits_all, class_logits_all = model(padded.unsqueeze(0))

    # last decoder block
    mask_logits = mask_logits_all[-1]
    class_logits = class_logits_all[-1]

    mask_logits = F.interpolate(mask_logits, IMG_SIZE, mode="bilinear")
    mask_logits = mask_logits[:, :, :new_h, :new_w]
    mask_logits = F.interpolate(mask_logits, (orig_h, orig_w), mode="bilinear")

    scores = class_logits[0].softmax(dim=-1)[:, :-1]
    all_labels = torch.arange(NUM_CLASSES, device=device).unsqueeze(0).repeat(scores.shape[0], 1).flatten()
    topk_scores, topk_idx = scores.flatten().topk(min(EVAL_TOP_K, scores.numel()), sorted=True)
    pred_labels = all_labels[topk_idx]
    query_idx = topk_idx // NUM_CLASSES
    ml = mask_logits[0][query_idx]

    masks_bool = ml > 0
    mask_scores = (ml.sigmoid().flatten(1) * masks_bool.flatten(1)).sum(1) / (masks_bool.flatten(1).sum(1) + 1e-6)
    final_scores = topk_scores * mask_scores

    keep = final_scores > SCORE_THRESH
    return masks_bool[keep].cpu().numpy(), pred_labels[keep].cpu().numpy(), final_scores[keep].cpu().numpy()

print("Helpers OK")

In [ ]:
# --- Collect test samples ---
samples = []
for img_path in sorted(IMG_DIR.glob("*.png")):
    label_path = LABEL_DIR / f"{img_path.stem}.txt"
    samples.append((img_path, label_path))

samples = samples[:NUM_SAMPLES]
print(f"Will visualize {len(samples)} test samples")

In [ ]:
# --- Visualize: Original | GT | Prediction ---
n = len(samples)
fig, axes = plt.subplots(n, 3, figsize=(18, 5 * n))
if n == 1:
    axes = axes[np.newaxis, :]

axes[0, 0].set_title("Original Image", fontsize=16, fontweight="bold")
axes[0, 1].set_title("Ground Truth", fontsize=16, fontweight="bold")
axes[0, 2].set_title("Prediction", fontsize=16, fontweight="bold")

for idx, (img_path, label_path) in enumerate(samples):
    print(f"\r  [{idx+1}/{n}] {img_path.name}", end="")

    pil_img = Image.open(img_path).convert("RGB")
    img_np = np.array(pil_img)
    h, w = img_np.shape[:2]
    img_tensor = tv_tensors.Image(pil_img)

    gt_masks, gt_labels = load_gt(label_path, h, w)
    pred_masks, pred_labels, pred_scores = predict(img_tensor)

    gt_overlay = render_masks_on_image(img_np, gt_masks, gt_labels)
    pred_overlay = render_masks_on_image(img_np, list(pred_masks), list(pred_labels))

    axes[idx, 0].imshow(img_np)
    axes[idx, 0].set_ylabel(img_path.stem, fontsize=10, rotation=0, labelpad=60, va="center")
    for ax in axes[idx]:
        ax.set_xticks([])
        ax.set_yticks([])
    axes[idx, 1].imshow(gt_overlay)
    axes[idx, 2].imshow(pred_overlay)

# Legend
legend_patches = [mpatches.Patch(color=CLASS_COLORS[i], label=CLASS_NAMES[i]) for i in range(NUM_CLASSES)]
fig.legend(handles=legend_patches, loc="upper center", ncol=NUM_CLASSES, fontsize=12,
           bbox_to_anchor=(0.5, 1.0), frameon=True)

plt.tight_layout(rect=[0, 0, 1, 0.98])
fig.savefig("test_visualization_50.png", dpi=80, bbox_inches="tight", facecolor="white")
print(f"\nSaved to test_visualization_50.png")
plt.show()